# 23 — Cluster-State Architecture for Measurement-Based Quantum Computation

**Engineering question**

How do frequency-mode graph resources become a programmable quantum computation resource?

Notebook 17 treated frequency modes as graph nodes and quantum correlations as graph edges.  
Notebook 23 moves from graph resource to computation:

```text
frequency-bin graph
→ cluster-state lattice
→ single-mode measurements
→ feed-forward
→ logical output
```

Engineering statement:

> Individual frequency-bin entanglement becomes a programmable computational graph once graph connectivity supports measurement-based processing.


## Setup

This notebook creates:

```text
figures/
results/csv/
results/json/
results/23_outputs.zip
```

The final cell supports both:

- **Google Colab**: starts a real browser download with `files.download(...)`
- **Jupyter**: displays a clickable `FileLink`


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Cluster-state resource pipeline

The notebooks now form a full engineering chain.

Notebook 11:

```text
Kerr physics generates frequency resources.
```

Notebook 13:

```text
Two-mode squeezing generates quantum resources.
```

Notebook 17:

```text
Frequency resources become graph resources.
```

Notebook 23:

```text
Graph resources become measurement-based computation.
```


In [ ]:
labels = [
    "Pump",
    "High-Q\nresonator",
    "Kerr χ³",
    "Frequency\ncomb",
    "Two-mode\nsqueezing",
    "Graph\nentanglement",
    "Cluster\nstate",
    "Single-mode\nmeasurements",
    "Logical\noutput",
]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(15, 3.4))

for xi, label in zip(x, labels):
    rect = plt.Rectangle((xi - 0.42, -0.25), 0.84, 0.5, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(xi, 0, label, ha="center", va="center", fontsize=9.4)

for i in range(len(labels) - 1):
    ax.annotate("", xy=(i + 0.58, 0), xytext=(i + 0.42, 0), arrowprops=dict(arrowstyle="->", linewidth=2))

ax.set_title("Cluster-State Resource Pipeline", fontsize=16)
ax.set_xlim(-0.7, len(labels) - 0.3)
ax.set_ylim(-0.65, 0.72)
ax.axis("off")

savefig(fig, "23_cluster_pipeline.png")
plt.show()


## 2. Cluster-state graph

A cluster state is not just many edges.

It is an entangled graph with geometry that supports measurement-based computation.

The graph below is an abstract lattice:

```text
nodes = optical frequency modes / encoded modes
edges = entanglement links
```

The computational value comes from connectivity plus measurement order.


In [ ]:
rows, cols = 4, 6
xs = np.arange(cols)
ys = np.arange(rows)

fig, ax = plt.subplots(figsize=(9.5, 5.5))

# edges
for y in ys:
    for x in xs[:-1]:
        ax.plot([x, x+1], [y, y], linewidth=2)
for x in xs:
    for y in ys[:-1]:
        ax.plot([x, x], [y, y+1], linewidth=2)

# selected diagonal bonds to suggest richer architecture
diags = [(0,0), (1,1), (2,0), (3,1), (1,2), (3,2)]
for x, y in diags:
    if x + 1 < cols and y + 1 < rows:
        ax.plot([x, x+1], [y, y+1], linewidth=1.4, alpha=0.6)

for y in ys:
    ax.scatter(xs, np.full_like(xs, y), s=140, zorder=3)

ax.text(cols/2 - 0.5, -0.55, "frequency-mode lattice / cluster-state graph", ha="center", fontsize=12)
ax.text(-0.6, rows/2 - 0.5, "measurement\npaths", ha="center", va="center", rotation=90, fontsize=11)

ax.set_title("Cluster States Encode Computation as Graph Connectivity", fontsize=16)
ax.set_aspect("equal")
ax.set_xlim(-0.8, cols - 0.2)
ax.set_ylim(-0.8, rows - 0.1)
ax.axis("off")

savefig(fig, "23_cluster_graph.png")
plt.show()


## 3. Measurement drives computation

Measurement-based quantum computation uses a prepared entangled resource.

The algorithm is not applied by directly wiring gates after state preparation.

Instead:

```text
prepare graph
→ measure selected nodes
→ update remaining logical state
```

The figure below shows a conceptual graph update after measurement.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.6))

def draw_chain(ax, measured=None, removed=False, title=""):
    ax.set_title(title, fontsize=13)
    ax.axis("off")
    x = np.arange(6)
    y = np.zeros(6)
    for i in range(5):
        if removed and (i == measured or i+1 == measured):
            continue
        ax.plot([x[i], x[i+1]], [0, 0], linewidth=2)
    for i in range(6):
        if removed and i == measured:
            ax.scatter([x[i]], [0], s=140, marker="x", linewidth=3)
        elif i == measured:
            ax.scatter([x[i]], [0], s=190, marker="s")
        else:
            ax.scatter([x[i]], [0], s=140)
        ax.text(x[i], -0.32, f"{i+1}", ha="center", fontsize=10)
    ax.set_xlim(-0.5, 5.5)
    ax.set_ylim(-0.8, 0.8)

draw_chain(axes[0], title="1. Prepare cluster")
draw_chain(axes[1], measured=2, title="2. Measure node")
draw_chain(axes[2], measured=2, removed=True, title="3. Logical state updates")

axes[1].text(2, 0.45, "measurement\nbasis", ha="center", fontsize=10)
axes[2].text(2, 0.45, "removed\nfrom graph", ha="center", fontsize=10)

fig.suptitle("Measurements Drive Computation Through the Graph", fontsize=16)

savefig(fig, "23_measurement_computation.png")
plt.show()


## 4. Measurement order and feed-forward

Measurement-based computation is temporal even when the cluster graph is prepared in advance.

A simplified sequence:

```text
prepare cluster
→ choose measurement basis
→ measure mode
→ feed-forward
→ choose next measurement basis
→ read logical output
```

Adaptive feed-forward is where hardware timing becomes part of the architecture.


In [ ]:
labels = [
    "Prepare\ncluster",
    "Measure\nmode 1",
    "Feed-forward",
    "Update\nbasis",
    "Measure\nmode 2",
    "Logical\noutput",
]

x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(12, 3.2))

ax.plot(x, np.zeros_like(x), linewidth=2)
ax.scatter(x, np.zeros_like(x), s=160, zorder=3)

for xi, label in zip(x, labels):
    ax.text(xi, -0.28, label, ha="center", va="top", fontsize=10)

for i in range(len(labels) - 1):
    ax.annotate("", xy=(i + 0.86, 0), xytext=(i + 0.14, 0), arrowprops=dict(arrowstyle="->", linewidth=1.8))

ax.text(2.5, 0.35, "adaptive measurement sequence", ha="center", fontsize=12)
ax.set_title("Computation Follows Measurement Order", fontsize=16)
ax.set_xlim(-0.6, len(labels) - 0.4)
ax.set_ylim(-0.7, 0.75)
ax.axis("off")

savefig(fig, "23_measurement_sequence.png")
plt.show()


## 5. Resource scaling

More modes alone do not guarantee useful computation.

A useful architecture depends on multiple scaling variables:

- usable modes,
- graph edges,
- measurement depth,
- squeezing quality,
- loss,
- feed-forward latency.

This plot is illustrative. It separates physical mode count from computational utility.


In [ ]:
modes = np.arange(4, 101, 4)
edges = 1.8 * modes
depth = np.sqrt(modes) * 6
usable = 0.68 * depth + 0.08 * edges

fig, ax = plt.subplots(figsize=(8.5, 5.2))

ax.plot(modes, modes, marker="o", linewidth=2.1, label="usable modes")
ax.plot(modes, edges, marker="s", linewidth=2.1, label="graph edges")
ax.plot(modes, usable, marker="^", linewidth=2.1, label="computational resource proxy")

ax.set_title("From Usable Modes to Computational Resources", fontsize=16)
ax.set_xlabel("Addressable frequency modes")
ax.set_ylabel("Relative resource scale")
ax.grid(True, alpha=0.3)
ax.legend()

ax.text(38, 45, "illustrative only:\nloss and squeezing decide usable scale", fontsize=10)

savefig(fig, "23_cluster_scaling.png")
plt.show()


## 6. Architecture constraints

Cluster-state computation introduces new engineering constraints.

The graph must be prepared.

The modes must be measured.

The measurement results must be processed quickly enough to choose later measurement bases.

This is where optical quantum hardware becomes an architecture problem.


In [ ]:
constraints = pd.DataFrame(
    [
        ["Squeezing", "Raises edge strength", "Insufficient squeezing limits fault tolerance"],
        ["Loss", "Preserves graph correlations", "Loss removes usable nodes and edges"],
        ["Mode control", "Selects measurement targets", "Poor control mixes channels"],
        ["Feed-forward", "Adapts later measurements", "Latency limits computation depth"],
        ["Detection", "Reads quadratures", "Detector inefficiency reduces correlation"],
    ],
    columns=["Constraint", "Architecture role", "Failure mode"],
)

fig, ax = plt.subplots(figsize=(12.5, 3.7))
ax.axis("off")

table = ax.table(
    cellText=constraints.values,
    colLabels=constraints.columns,
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(9.8)
table.scale(1.15, 1.72)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Architecture Constraints for Measurement-Based Quantum Computation", fontsize=16, pad=20)

savefig(fig, "23_constraints.png")
plt.show()


## 7. Engineering summary

The notebook's main shift is:

```text
entangled graph
→ computational substrate
```

That shift requires measurements, feed-forward, and fault-tolerance thinking.


In [ ]:
summary = pd.DataFrame(
    [
        {
            "Resource": "Frequency mode",
            "Physical meaning": "Optical channel",
            "Computational role": "Graph node",
            "Next engineering challenge": "Stabilization",
        },
        {
            "Resource": "Two-mode squeezing",
            "Physical meaning": "Pair resource",
            "Computational role": "Graph edge",
            "Next engineering challenge": "Higher squeezing",
        },
        {
            "Resource": "Graph connectivity",
            "Physical meaning": "Entanglement lattice",
            "Computational role": "Computational substrate",
            "Next engineering challenge": "Programmability",
        },
        {
            "Resource": "Measurement",
            "Physical meaning": "Quadrature readout",
            "Computational role": "Executes algorithm",
            "Next engineering challenge": "Fast feed-forward",
        },
        {
            "Resource": "Cluster state",
            "Physical meaning": "Multipartite entangled resource",
            "Computational role": "Measurement-based quantum computer",
            "Next engineering challenge": "Fault tolerance",
        },
    ]
)

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.axis("off")

table = ax.table(
    cellText=summary.values,
    colLabels=summary.columns,
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(9.4)
table.scale(1.08, 1.68)

for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")

ax.set_title("Engineering Summary: Cluster States Convert Graphs into Computation", fontsize=16, pad=20)

savefig(fig, "23_engineering_summary.png")

summary.to_csv(CSV / "23_summary.csv", index=False)
summary.to_json(JS / "23_summary.json", orient="records", indent=2)

plt.show()
summary


## 8. Export and download

This cell packages all outputs and starts a download.

It uses Colab's native downloader when available.  
For local Jupyter, it falls back to a clickable `FileLink`.


In [ ]:
zip_path = RES / "23_outputs.zip"

files_to_zip = [
    FIG / "23_cluster_pipeline.png",
    FIG / "23_cluster_graph.png",
    FIG / "23_measurement_computation.png",
    FIG / "23_measurement_sequence.png",
    FIG / "23_cluster_scaling.png",
    FIG / "23_constraints.png",
    FIG / "23_engineering_summary.png",
    CSV / "23_summary.csv",
    JS / "23_summary.json",
]

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))

print(f"Created: {zip_path}")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))

summary


## Takeaways

- Cluster states convert entangled frequency graphs into a computational substrate.
- Computation is performed by measurement order, basis choice, and feed-forward.
- Larger frequency combs can support larger graph resources, but useful computation depends on squeezing, loss, detection, and timing.
- This is the bridge from microcomb quantum resources to measurement-based quantum computing.

## Next notebook

**29 — Loss and Measurement Constraints**

Notebook 23 describes the architecture.

Notebook 29 should ask what prevents it from scaling cleanly:

```text
cluster graph
→ loss channels
→ finite squeezing
→ detector inefficiency
→ feed-forward latency
→ usable computation window
```
